# Policy Gradient: REINFORCE with cart pole balancing

In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import gymnasium as gym
import numpy as np

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size):
        super().__init__()
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_size = hidden_size
        self.layer1 = nn.Linear(self.state_size, self.hidden_size)
        self.layer2 = nn.Linear(self.hidden_size, self.hidden_size)
        self.output = nn.Linear(self.hidden_size, self.action_size)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return F.softmax(self.output(x), dim=-1)

In [ ]:
class CartPoleAgent:
    def __init__(self, state_size, action_size, hidden_size, learning_rate, gamma):
        self.action_size = action_size
        self.policy_net = PolicyNetwork(state_size, action_size, hidden_size)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=learning_rate)
        self.gamma = gamma

    def select_action(self, state):
        with torch.no_grad():
            input_state = torch.tensor(state, dtype=torch.float32).to(device)
            action_probs = self.policy_net(input_state)
            action_probs = action_probs.detach().cpu().numpy()
            action = np.random.choice(np.arange(self.action_size), p=action_probs)
        return action


    def discount_and_normalize_rewards(self, reward_list):
        traj_len = len(reward_list)
        return_array = np.zeros_like(reward_list)
        reward_to_go = 0.0

        for i in range(traj_len -1, -1, -1):
            reward_to_go = reward_to_go * self.gamma + reward_list[i]
            return_array[i] = reward_to_go

        return_array -= np.mean(return_array)
        return_array /= np.std(return_array)

        return return_array



    def train(self, state_list, action_list, reward_list):
        return_array = self.discount_and_normalize_rewards(reward_list)
        state_t = torch.tensor(state_list, dtype=torch.float32).to(device)
        action_t = torch.tensor(action_list, dtype=torch.float32).to(device).view(-1, 1)
        return_t = torch.tensor(return_array, dtype=torch.float32).to(device).view(-1, 1)
        selected_action_prob = self.policy_net(state_t).gather(1, action_t)
        loss = torch.mean(-torch.log(selected_action_prob) * return_t)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.detach().cpu().numpy()



        